# Step 0 — Setup

In [1]:
# Step 0: Imports
import os
import json
import numpy as np
import pandas as pd
import requests
from pathlib import Path

# Reproducibility
SEED = 42
rng = np.random.default_rng(SEED)

DATA_DIR = Path("zenodo_2667859")
DATA_DIR.mkdir(exist_ok=True)

# Step 1 — Download Zenodo record files programmatically

In [2]:
# Step 1: Download Zenodo record files
ZENODO_RECORD_ID = 2667859
api_url = f"https://zenodo.org/api/records/{ZENODO_RECORD_ID}"

resp = requests.get(api_url)
resp.raise_for_status()
record = resp.json()

print("Title:", record.get("metadata", {}).get("title"))
print("Files found:", len(record.get("files", [])))

def download_file(url, out_path, chunk_size=1024*1024):
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done = 0
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    done += len(chunk)
    return total, done

for f in record["files"]:
    fname = f["key"]
    file_url = f["links"]["self"]  # direct download endpoint
    out_path = DATA_DIR / fname
    if out_path.exists():
        print(f"✓ Already downloaded: {fname}")
        continue
    total, done = download_file(file_url, out_path)
    print(f"↓ Downloaded: {fname} ({done/1e6:.2f} MB)")

Title: Reddit C-SSRS Suicide Dataset
Files found: 5
↓ Downloaded: suicidal_attempt.csv (0.00 MB)
↓ Downloaded: suicidal_indicator.csv (0.05 MB)
↓ Downloaded: suicidal_ideation.csv (0.01 MB)
↓ Downloaded: suicidal_behavior.csv (0.00 MB)
↓ Downloaded: 500_Reddit_users_posts_labels.csv (3.62 MB)


# Step 2 — Load the main labeled dataset

In [3]:
# Step 2: Load main dataset
main_path = DATA_DIR / "500_Reddit_users_posts_labels.csv"
df = pd.read_csv(main_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df["Label"].value_counts(dropna=False))
df.head(3)

Shape: (500, 3)
Columns: ['User', 'Post', 'Label']
Label
Ideation      171
Supportive    108
Indicator      99
Behavior       77
Attempt        45
Name: count, dtype: int64


,User,Post,Label
0,user-0,"['Its not a viable option, and youll be leavin...",Supportive
1,user-1,['It can be hard to appreciate the notion that...,Ideation
2,user-2,"['Hi, so last night i was sitting on the ledge...",Behavior


# Step 3 — Map Zenodo labels → tiers (0–4)
keep this mapping flexible because label names sometimes differ by capitalization/spaces.

In [4]:
# Step 3: Normalize labels and map to tiers

def normalize_label(x):
    if pd.isna(x):
        return "unknown"
    return str(x).strip().lower()

df["label_norm"] = df["Label"].apply(normalize_label)

# Common 5-class scheme in this dataset:
# supportive, indicator, ideation, behavior, attempt
tier_map = {
    "supportive": 0,
    "indicator": 1,
    "ideation": 2,
    "behavior": 3,
    "attempt": 4
}

# If there are unexpected labels, they will become NaN -> we’ll drop them.
df["tier"] = df["label_norm"].map(tier_map)

print("Unmapped labels:", df.loc[df["tier"].isna(), "Label"].unique())
df = df.dropna(subset=["tier"]).copy()
df["tier"] = df["tier"].astype(int)

print(df["tier"].value_counts().sort_index())

Unmapped labels: []
tier
0    108
1     99
2    171
3     77
4     45
Name: count, dtype: int64


# Step 4 — Define principled conditional generators

We generate structured C-SSRS-like features conditioned on tier, using:

Bernoulli probabilities for presence/behaviors

Categorical distributions for ideation severity

Beta distributions (scaled) for intensity subscales

Bucket-mixtures for temporal variables (acute + persistence)

In [5]:
# Step 4: Conditional distributions (tunable)

# 4.1 Ideation presence probability by tier
p_I_present = {0: 0.05, 1: 0.35, 2: 0.85, 3: 0.95, 4: 0.98}

# 4.2 Ideation severity categorical (given I_present=1)
# keys are tiers, values are dict(level -> prob)
sev_dist = {
    0: {0: 1.0},  # supportive -> no ideation
    1: {1: 0.35, 2: 0.40, 3: 0.20, 4: 0.05},
    2: {2: 0.30, 3: 0.40, 4: 0.30},
    3: {3: 0.20, 4: 0.50, 5: 0.30},
    4: {4: 0.40, 5: 0.60},
}

# 4.3 Intensity beta params per tier (scaled to 1..5)
beta_params = {
    0: (1.2, 5.0),
    1: (1.8, 3.8),
    2: (2.6, 2.6),
    3: (3.5, 2.0),
    4: (4.5, 1.6),
}

def sample_intensity(tier, size=1):
    a, b = beta_params[tier]
    x = rng.beta(a, b, size=size)           # 0..1
    s = 1 + np.rint(4 * x).astype(int)      # 1..5
    return np.clip(s, 1, 5)

def sample_categorical(dist_dict):
    levels = np.array(list(dist_dict.keys()))
    probs  = np.array(list(dist_dict.values()), dtype=float)
    probs  = probs / probs.sum()
    return int(rng.choice(levels, p=probs))

# 4.4 Behavior probabilities by tier (can be made hierarchical)
p_B_preparatory = {0:0.01, 1:0.05, 2:0.10, 3:0.25, 4:0.35}
p_B_aborted     = {0:0.00, 1:0.02, 2:0.06, 3:0.20, 4:0.25}
p_B_interrupted = {0:0.00, 1:0.02, 2:0.05, 3:0.22, 4:0.30}
p_B_attempt     = {0:0.00, 1:0.01, 2:0.02, 3:0.10, 4:0.70}

# 4.5 Temporal bucket mixtures
# days_since_last_event buckets: (low, high) ranges in days
EVENT_BUCKETS = [(0,7), (8,30), (31,3650)]
# mixture weights by tier
event_mix = {
    0: [0.00, 0.00, 1.00],  # supportive -> no event; we’ll handle separately
    1: [0.15, 0.35, 0.50],
    2: [0.30, 0.40, 0.30],
    3: [0.45, 0.40, 0.15],
    4: [0.60, 0.30, 0.10],
}

# duration_since_onset buckets
ONSET_BUCKETS = [(0,7), (8,30), (31,90), (91,3650)]
onset_mix = {
    0: [1.00, 0.00, 0.00, 0.00],
    1: [0.40, 0.35, 0.20, 0.05],
    2: [0.20, 0.40, 0.25, 0.15],
    3: [0.20, 0.35, 0.30, 0.15],
    4: [0.40, 0.30, 0.20, 0.10],  # attempt can be acute or chronic
}

def sample_from_buckets(buckets, weights):
    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum()
    idx = int(rng.choice(len(buckets), p=weights))
    lo, hi = buckets[idx]
    return int(rng.integers(lo, hi+1))

# Step 5 — Generate the synthetic structured R¹⁵ dataset

This creates the exact Task 3 structured features:

age_group

ideation + intensity

behavior

temporal (days_since_last_event, duration_since_onset)

In [17]:
# Step 5 (UPDATED): Generate synthetic structured dataset (R15)

def sample_age_group(size, teen_prior=0.15):
    return np.where(rng.random(size) < teen_prior, "Teen", "Adult")

n = len(df)

out = pd.DataFrame({
    "user": df["User"],
    "post_text": df["Post"],
    "label": df["Label"],
    "tier": df["tier"],
    "age_group": sample_age_group(n, teen_prior=0.15)
})

# Initialize arrays
I_present = np.zeros(n, dtype=int)
I_sev     = np.zeros(n, dtype=int)

I_freq = np.zeros(n, dtype=int)
I_dur  = np.zeros(n, dtype=int)
I_ctrl = np.zeros(n, dtype=int)
I_det  = np.zeros(n, dtype=int)
I_reas = np.zeros(n, dtype=int)

B_prep = np.zeros(n, dtype=int)
B_abort= np.zeros(n, dtype=int)
B_intr = np.zeros(n, dtype=int)
B_att  = np.zeros(n, dtype=int)
B_any  = np.zeros(n, dtype=int)

days_since_last_event = np.zeros(n, dtype=int)
duration_since_onset  = np.zeros(n, dtype=int)

for i, t in enumerate(out["tier"].values):
    # --- Ideation presence ---
    I_present[i] = int(rng.random() < p_I_present[t])

    # --- Ideation severity + intensity (if ideation present) ---
    if I_present[i] == 1:
        I_sev[i] = sample_categorical(sev_dist[t])
        I_freq[i] = sample_intensity(t)[0]
        I_dur[i]  = sample_intensity(t)[0]
        I_ctrl[i] = sample_intensity(t)[0]
        I_det[i]  = sample_intensity(t)[0]
        I_reas[i] = sample_intensity(t)[0]
    else:
        I_sev[i] = 0
        I_freq[i] = I_dur[i] = I_ctrl[i] = I_det[i] = I_reas[i] = 0

    # --- Behavior indicators (initial sampling) ---
    B_prep[i]  = int(rng.random() < p_B_preparatory[t])
    B_abort[i] = int(rng.random() < p_B_aborted[t])
    B_intr[i]  = int(rng.random() < p_B_interrupted[t])
    B_att[i]   = int(rng.random() < p_B_attempt[t])

    # ==========================================================
    # HARD CONSTRAINTS BY TIER (keeps synthetic data consistent)
    # ==========================================================
    if t == 4:  # Attempt tier: must have actual attempt + ideation
        B_att[i] = 1
        I_present[i] = 1

    elif t == 3:  # Behavior tier: must have some behavior + ideation
        if (B_prep[i] + B_abort[i] + B_intr[i] + B_att[i]) == 0:
            # force one behavior type (not necessarily attempt)
            choice = rng.choice(["prep", "abort", "intr"], p=[0.4, 0.3, 0.3])
            if choice == "prep":
                B_prep[i] = 1
            elif choice == "abort":
                B_abort[i] = 1
            else:
                B_intr[i] = 1
        I_present[i] = 1

    elif t == 2:  # Ideation tier
        I_present[i] = 1
        # Ideation tier should not have behavior
        B_prep[i] = 0
        B_abort[i] = 0
        B_intr[i] = 0
        B_att[i] = 0

    # Recompute B_any after constraints
    B_any[i] = int((B_prep[i] or B_abort[i] or B_intr[i] or B_att[i]))

    # Ensure attempt implies high ideation severity + intensity present
    if B_att[i] == 1:
        I_present[i] = 1
        if I_sev[i] < 4:
            I_sev[i] = sample_categorical({4: 0.4, 5: 0.6})
        # fill intensity if it was zero (because ideation was sampled absent earlier)
        if I_freq[i] == 0:
            tier_for_intensity = max(t, 3)
            I_freq[i] = sample_intensity(tier_for_intensity)[0]
            I_dur[i]  = sample_intensity(tier_for_intensity)[0]
            I_ctrl[i] = sample_intensity(tier_for_intensity)[0]
            I_det[i]  = sample_intensity(tier_for_intensity)[0]
            I_reas[i] = sample_intensity(tier_for_intensity)[0]

    # --- Temporal features ---
    if (I_present[i] == 1) or (B_any[i] == 1):
        days_since_last_event[i] = sample_from_buckets(EVENT_BUCKETS, event_mix[t])
        duration_since_onset[i]  = sample_from_buckets(ONSET_BUCKETS, onset_mix[t])

        # Coherence: onset duration should generally be >= last event
        if duration_since_onset[i] < days_since_last_event[i]:
            duration_since_onset[i] = min(days_since_last_event[i] + int(rng.integers(0, 30)), 3650)
    else:
        days_since_last_event[i] = 0
        duration_since_onset[i]  = 0

# Assign to dataframe
out["I_present"] = I_present
out["I_severity_level"] = I_sev
out["I_freq"] = I_freq
out["I_duration"] = I_dur
out["I_controllability"] = I_ctrl
out["I_deterrents"] = I_det
out["I_reasons"] = I_reas

out["B_preparatory"] = B_prep
out["B_aborted"] = B_abort
out["B_interrupted"] = B_intr
out["B_actual_attempt"] = B_att
out["B_any"] = B_any

out["days_since_last_event"] = days_since_last_event
out["duration_since_onset"] = duration_since_onset

out.head(3)

,user,post_text,label,tier,age_group,I_present,I_severity_level,I_freq,I_duration,I_controllability,I_deterrents,I_reasons,B_preparatory,B_aborted,B_interrupted,B_actual_attempt,B_any,days_since_last_event,duration_since_onset
0,user-0,"['Its not a viable option, and youll be leavin...",Supportive,0,Adult,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,user-1,['It can be hard to appreciate the notion that...,Ideation,2,Adult,1,4,3,4,3,2,2,0,0,0,0,0,3,55
2,user-2,"['Hi, so last night i was sitting on the ledge...",Behavior,3,Adult,1,4,4,3,3,3,4,1,0,0,0,1,4,52


# Step 6 — Compute Task 3 severity score (0–100) with dual temporal module

This implements:

Ideation (0–35)

Intensity (0–20)

Behavior (0–30, max-hierarchy)

Temporal (0–15) = Acute(0–8) + Persistence(0–7)

In [18]:
# Step 6 (UPDATED): Task 3 severity scoring (rule-based + dual temporal module)

def temporal_points(days_since_last_event, duration_since_onset):
    # Acute: 0 means "today" → full acute boost
    if 0 <= days_since_last_event <= 7:
        acute = 8
    elif 8 <= days_since_last_event <= 30:
        acute = 4
    else:
        acute = 0

    # Persistence: longer duration => more points
    if duration_since_onset <= 7:
        persist = 0
    elif duration_since_onset <= 30:
        persist = 2
    elif duration_since_onset <= 90:
        persist = 4
    else:
        persist = 7

    return acute, persist, acute + persist

def behavior_points(row):
    # Hierarchical max (dominant behavior wins)
    pts = 0
    if row["B_preparatory"] == 1:      pts = max(pts, 8)
    if row["B_aborted"] == 1:          pts = max(pts, 12)
    if row["B_interrupted"] == 1:      pts = max(pts, 18)
    if row["B_actual_attempt"] == 1:   pts = max(pts, 30)
    return float(pts)

def intensity_points(row):
    if row["I_present"] == 0:
        return 0.0
    sub = np.array([
        row["I_freq"],
        row["I_duration"],
        row["I_controllability"],
        row["I_deterrents"],
        row["I_reasons"]
    ], dtype=float)

    # Normalize 1..5 -> 0..1
    norm = (sub - 1.0) / 4.0
    return float(20.0 * norm.mean())

def ideation_points(row):
    return float(7.0 * row["I_severity_level"])  # 0..35

def risk_band(score, actual_attempt):
    if actual_attempt == 1 or score >= 85:
        return "Critical"
    elif score >= 65:
        return "High"
    elif score >= 35:
        return "Medium"
    else:
        return "Low"

# Compute points
out["points_behavior"] = out.apply(behavior_points, axis=1)
out["points_ideation"] = out.apply(ideation_points, axis=1)
out["points_intensity"] = out.apply(intensity_points, axis=1)

acute_list = []
persist_list = []
temp_list = []

acute_list = []
persist_list = []
temp_list = []

for idx, row in out.iterrows():
    if row["I_present"] == 1 or row["B_any"] == 1:
        a, p, tp = temporal_points(
            int(row["days_since_last_event"]),
            int(row["duration_since_onset"])
        )
    else:
        a, p, tp = 0, 0, 0

    acute_list.append(a)
    persist_list.append(p)
    temp_list.append(tp)

out["points_acute"] = acute_list
out["points_persistence"] = persist_list
out["points_temporal"] = temp_list

# Final score
out["severity_score_rule"] = (
    out["points_ideation"] +
    out["points_intensity"] +
    out["points_behavior"] +
    out["points_temporal"]
).clip(0, 100)

out["risk_band"] = [
    risk_band(s, a) for s, a in zip(out["severity_score_rule"].values, out["B_actual_attempt"].values)
]

# Quick view
out[["label","tier","severity_score_rule","risk_band",
     "points_ideation","points_intensity","points_behavior","points_temporal"]].head(10)

,label,tier,severity_score_rule,risk_band,points_ideation,points_intensity,points_behavior,points_temporal
0,Supportive,0,0.0,Low,0.0,0.0,0.0,0
1,Ideation,2,49.0,Medium,28.0,9.0,0.0,12
2,Behavior,3,60.0,Medium,28.0,12.0,8.0,12
3,Attempt,4,79.0,Critical,28.0,13.0,30.0,8
4,Ideation,2,43.0,Medium,21.0,10.0,0.0,12
5,Supportive,0,0.0,Low,0.0,0.0,0.0,0
6,Supportive,0,0.0,Low,0.0,0.0,0.0,0
7,Ideation,2,45.0,Medium,28.0,11.0,0.0,6
8,Supportive,0,12.0,Low,0.0,5.0,0.0,7
9,Ideation,2,50.0,Medium,28.0,10.0,0.0,12


In [19]:
# Step 7: Save final dataset
OUT_PATH = "task3_synthetic_cssrs_structured.csv"
out.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

Saved: task3_synthetic_cssrs_structured.csv


In [20]:
# Optional sanity checks
print(out["risk_band"].value_counts())
print(out.groupby("label")["severity_score_rule"].describe())

# Ensure monotonic-ish trend by tier (not perfect, but should trend upward)
print(out.groupby("tier")["severity_score_rule"].mean().sort_index())

risk_band
Low         252
Medium      160
Critical     59
High         29
Name: count, dtype: int64
            count       mean        std   min   25%   50%   75%   max
label                                                                
Attempt      45.0  86.377778   4.473604  76.0  83.0  87.0  90.0  94.0
Behavior     77.0  66.220779  12.320953  11.0  61.0  65.0  72.0  92.0
Ideation    171.0  36.327485  13.416334   1.0  32.5  40.0  45.0  56.0
Indicator    99.0   9.000000  13.272651   0.0   0.0   0.0  24.0  40.0
Supportive  108.0   0.750000   2.923079   0.0   0.0   0.0   0.0  15.0
tier
0     0.750000
1     9.000000
2    36.327485
3    66.220779
4    86.377778
Name: severity_score_rule, dtype: float64
